# ABS wheel-speed forecasting — complete training pipeline

This notebook performs the complete workflow:

1. locate the MATLAB dataset and manifest;
2. prepare or reuse the common HDF5 cache;
3. load train, validation and test partitions;
4. train CNN, GRU and LSTM with the same `Trainer`;
5. compare their validation and monitored-test metrics.

The test partition is monitored during training as requested, but checkpoint selection, learning-rate scheduling and early stopping depend only on validation.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent.parent
elif cwd.name == "model_training":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd


sys.path.insert(0, str(PROJECT_ROOT))

from model_training.data import create_dataloaders, prepare_hdf5_dataset
from model_training.models import CNNForecaster, GRUForecaster, LSTMForecaster
from model_training.monitoring import plot_histories
from model_training.trainer import Trainer

SIMULATION_RESULTS = PROJECT_ROOT / "ABS_SoH_Simulator" / "simulation_results"
MODEL_TRAINING = PROJECT_ROOT / "model_training"
CACHE_DIRECTORY = MODEL_TRAINING / "cache"
EXPERIMENTS_DIRECTORY = MODEL_TRAINING / "experiments"

CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Experiment parameters

These are simple notebook variables, not a configuration class.

In [ ]:
HISTORY_LENGTH = 20
HORIZON = 5
STRIDE = 2

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
SPLIT_SEED = 42

BATCH_SIZE = 1024  # Aligned with the HDF5 chunk size.
NUM_WORKERS = 0  # One vectorized HDF5 read per batch; no worker overhead.
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 8

# Test metrics are computed every epoch. Set a positive integer to monitor
# only a fixed number of test batches during development, or None for all.
MAX_TEST_BATCHES = 100
PROGRESS_EVERY = 100

HDF5_CACHE = CACHE_DIRECTORY / "wheel_speed_windows.h5"
SPLIT_CSV = CACHE_DIRECTORY / "simulation_splits.csv"

## 3. Locate the generated MATLAB data

The newest dataset and manifest are selected automatically. Replace the paths manually here if a specific campaign must be used.

In [ ]:
dataset_candidates = sorted(
    SIMULATION_RESULTS.glob("abs_healthy_braking_dataset_*.csv")
)
manifest_candidates = sorted(
    SIMULATION_RESULTS.glob("abs_healthy_braking_manifest_*.csv")
)

assert dataset_candidates, (
    "No braking dataset was found. Run BrakingDatasetGenerator in MATLAB first."
)
assert manifest_candidates, "No braking dataset manifest was found."

DATASET_CSV = dataset_candidates[-1]
MANIFEST_CSV = manifest_candidates[-1]

print("Dataset:", DATASET_CSV)
print("Manifest:", MANIFEST_CSV)
print("HDF5 cache:", HDF5_CACHE)

## 4. Prepare or reuse the common HDF5 dataset

The large CSV is converted only once. If the cache already exists, it is reused. To prepare a different campaign, change `HDF5_CACHE` to a new filename.

In [ ]:
if HDF5_CACHE.exists():
    print("Existing HDF5 cache reused:", HDF5_CACHE)
    summary_path = HDF5_CACHE.with_suffix(".summary.json")
    if summary_path.exists():
        print(summary_path.read_text(encoding="utf-8"))
else:
    preparation_summary = prepare_hdf5_dataset(
        DATASET_CSV,
        MANIFEST_CSV,
        HDF5_CACHE,
        SPLIT_CSV,
        history_length=HISTORY_LENGTH,
        horizon=HORIZON,
        stride=STRIDE,
        train_ratio=TRAIN_RATIO,
        validation_ratio=VALIDATION_RATIO,
        test_ratio=TEST_RATIO,
        seed=SPLIT_SEED,
    )
    preparation_summary

## 5. Load the three partitions

In [ ]:
train_loader, validation_loader, test_loader = create_dataloaders(
    HDF5_CACHE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print("Train windows:", len(train_loader.dataset))
print("Validation windows:", len(validation_loader.dataset))
print("Test windows:", len(test_loader.dataset))

batch = next(iter(train_loader))
print("X batch shape:", tuple(batch["x"].shape))
print("Y batch shape:", tuple(batch["y"].shape))

## 6. Create the unified Trainer

In [ ]:
trainer = Trainer(
    train_loader,
    validation_loader,
    test_loader,
    output_directory=EXPERIMENTS_DIRECTORY,
)

print("Training device:", trainer.device)

## 7. Train the CNN

In [ ]:
model = CNNForecaster(
    history_length=HISTORY_LENGTH,
    horizon=HORIZON,
)

cnn_history = trainer.train(
    model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=EARLY_STOPPING_PATIENCE,
    run_name="cnn",
    test_every=1,
    max_test_batches=MAX_TEST_BATCHES,
    progress_every=PROGRESS_EVERY,
)
cnn_test_metrics = trainer.evaluate_test(model)
cnn_test_metrics

## 8. Train the GRU

In [ ]:
model = GRUForecaster(
    history_length=HISTORY_LENGTH,
    horizon=HORIZON,
)

gru_history = trainer.train(
    model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=EARLY_STOPPING_PATIENCE,
    run_name="gru",
    test_every=1,
    max_test_batches=MAX_TEST_BATCHES,
    progress_every=PROGRESS_EVERY,
)
gru_test_metrics = trainer.evaluate_test(model)
gru_test_metrics

## 9. Train the LSTM

In [ ]:
model = LSTMForecaster(
    history_length=HISTORY_LENGTH,
    horizon=HORIZON,
)

lstm_history = trainer.train(
    model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=EARLY_STOPPING_PATIENCE,
    run_name="lstm",
    test_every=1,
    max_test_batches=MAX_TEST_BATCHES,
    progress_every=PROGRESS_EVERY,
)
lstm_test_metrics = trainer.evaluate_test(model)
lstm_test_metrics

## 10. Compare the three models

In [ ]:
histories = {
    "CNN": cnn_history,
    "GRU": gru_history,
    "LSTM": lstm_history,
}
plot_histories(histories)

In [ ]:
rows = []
for name, history in histories.items():
    best = history.loc[history["validation_loss"].idxmin()]
    rows.append({
        "model": name,
        "best_epoch": int(best["epoch"]),
        "validation_mae_mps": best["validation_mae"],
        "validation_rmse_mps": best["validation_rmse"],
        "test_mae_monitor_mps": best["test_mae"],
        "test_rmse_monitor_mps": best["test_rmse"],
        "epoch_seconds": best["epoch_seconds"],
    })

comparison = pd.DataFrame(rows).sort_values("validation_mae_mps")
comparison

## 8. Export the selected GRU to ONNX for MATLAB

The exported network includes input normalization and output denormalization. MATLAB therefore supplies raw wheel speeds in m/s with shape `[batch, 20, 4]` and receives predictions in m/s with shape `[batch, 5, 4]`, using wheel order `FL, FR, RL, RR`.

In [ ]:
import json

try:
    import onnx
except ImportError as exc:
    raise ImportError(
        "ONNX is required only for export. Install it once with: %pip install onnx"
    ) from exc


class GRUForMatlab(torch.nn.Module):
    """Expose the trained GRU directly in physical units (m/s)."""

    def __init__(self, predictor, mean, std):
        super().__init__()
        self.predictor = predictor
        self.register_buffer("normalization_mean", torch.as_tensor(mean).view(1, 1, 4))
        self.register_buffer("normalization_std", torch.as_tensor(std).view(1, 1, 4))

    def forward(self, wheel_speed_history_mps):
        normalized_history = (
            wheel_speed_history_mps - self.normalization_mean
        ) / self.normalization_std
        normalized_forecast = self.predictor(normalized_history)
        return (
            normalized_forecast * self.normalization_std
            + self.normalization_mean
        )


gru_run_directory = EXPERIMENTS_DIRECTORY / "gru"
checkpoint_path = gru_run_directory / "best_model.pt"
metadata_path = gru_run_directory / "run_metadata.json"
assert checkpoint_path.exists(), f"Missing checkpoint: {checkpoint_path}"
assert metadata_path.exists(), f"Missing metadata: {metadata_path}"

gru_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

selected_gru = GRUForecaster(
    history_length=int(gru_metadata["history_length"]),
    horizon=int(gru_metadata["horizon"]),
)
selected_gru.load_state_dict(checkpoint["model_state"])

matlab_gru = GRUForMatlab(
    selected_gru,
    gru_metadata["normalization_mean"],
    gru_metadata["normalization_std"],
).eval()

export_directory = MODEL_TRAINING / "exports"
export_directory.mkdir(parents=True, exist_ok=True)
onnx_path = export_directory / "abs_wheel_speed_gru.onnx"

example_history_mps = torch.zeros(
    1, int(gru_metadata["history_length"]), 4, dtype=torch.float32
)
with torch.no_grad():
    torch.onnx.export(
        matlab_gru,
        example_history_mps,
        onnx_path,
        input_names=["wheel_speed_history_mps"],
        output_names=["wheel_speed_forecast_mps"],
        dynamic_axes={
            "wheel_speed_history_mps": {0: "batch_size"},
            "wheel_speed_forecast_mps": {0: "batch_size"},
        },
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("Validated ONNX model:", onnx_path)
print("Input : [batch, 20, 4] raw m/s (FL, FR, RL, RR)")
print("Output: [batch, 5, 4] predicted m/s (FL, FR, RL, RR)")

In [ ]:
%pip install onnx